# 🧩 03 — Customer Segmentation (RFM Analysis)
**E-Commerce Customer & Revenue Analytics Platform**

This notebook implements RFM (Recency, Frequency, Monetary) analysis:
- Compute RFM metrics per customer
- Assign quantile-based RFM scores (1–4)
- Segment customers: Champions, Loyal, Potential Loyalists, At Risk, Lost
- Generate visualisations for each segment

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

CLEAN_PATH = os.path.join('..', 'data', 'processed', 'cleaned_data.csv')
df = pd.read_csv(CLEAN_PATH, parse_dates=['InvoiceDate'])
print(f'Loaded: {df.shape[0]:,} rows')

## Step 1: Compute RFM Metrics

In [ ]:
from customer_segmentation import compute_rfm, score_rfm, assign_segment

rfm = compute_rfm(df)
print(f'RFM table: {rfm.shape}')
rfm.head(10)

In [ ]:
print('=== RFM Metric Summary ===')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))

## Step 2: Score & Segment Customers

In [ ]:
rfm = score_rfm(rfm)
rfm = assign_segment(rfm)
print('\nSegment distribution:')
print(rfm['Segment'].value_counts())

## Step 3: Segment Distribution Chart

In [ ]:
COLORS = {
    'Champions': '#6C63FF',
    'Loyal Customers': '#43B6C8',
    'Potential Loyalists': '#56D79E',
    'At Risk': '#FFB347',
    'Lost Customers': '#FF6B6B'
}

counts = rfm['Segment'].value_counts().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# Horizontal bar
colors_bar = [COLORS.get(s, '#aaa') for s in counts.index]
axes[0].barh(counts.index, counts.values, color=colors_bar, edgecolor='white', height=0.6)
for i, (val, label) in enumerate(zip(counts.values, counts.index)):
    axes[0].text(val + 3, i, f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Customer Count by Segment', fontweight='bold', pad=10)
axes[0].set_xlabel('Number of Customers')
axes[0].set_facecolor('#f8f9fa')

# Donut
vc = rfm['Segment'].value_counts()
donut_colors = [COLORS.get(s, '#aaa') for s in vc.index]
axes[1].pie(vc.values, labels=vc.index, autopct='%1.1f%%',
            colors=donut_colors, startangle=90,
            wedgeprops=dict(width=0.55, edgecolor='white'))
axes[1].set_title('Segment Share (%)', fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', 'rfm_segment_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Frequency vs Monetary Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')

for seg, grp in rfm.groupby('Segment'):
    ax.scatter(grp['Frequency'], grp['Monetary'],
               label=seg, alpha=0.65, s=40,
               color=COLORS.get(seg, '#aaa'), edgecolors='none')

ax.set_title('RFM: Frequency vs Monetary Value by Segment',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Order Frequency')
ax.set_ylabel('Monetary Value (£)')
ax.legend(title='Segment', bbox_to_anchor=(1.01, 1))
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', 'rfm_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Revenue Contribution by Segment

In [ ]:
seg_rev = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')
bar_colors = [COLORS.get(s, '#aaa') for s in seg_rev.index]
bars = ax.bar(seg_rev.index, seg_rev.values, color=bar_colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, seg_rev.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'£{val:,.0f}', ha='center', va='bottom', fontsize=9)
ax.set_title('Total Revenue Contribution by Customer Segment',
             fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Revenue (£)')
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', 'rfm_segment_revenue.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 6: Export Segmentation Results

In [ ]:
out_path = os.path.join('..', 'data', 'processed', 'rfm_segments.csv')
rfm.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'\nTop Champions by Monetary Value:')
rfm[rfm['Segment'] == 'Champions'].sort_values('Monetary', ascending=False).head(10)

## ✅ Summary

| Segment | Description |
|---------|-------------|
| **Champions** | High R, F, M — best customers |
| **Loyal Customers** | Recent, frequent buyers |
| **Potential Loyalists** | Recent with moderate spend |
| **At Risk** | Not recent, but historically frequent |
| **Lost Customers** | Low recency, frequency, and spend |

**Next**: `04_forecasting.ipynb`